# Stage 2 - Scenario Generation

This notebook uploads a zipped input bundle, loads the fine-tuned adapter, and generates both a Playwright JSON file and a human-readable scenario summary.

If you run this from VS Code with the Google Colab extension:
- connect the notebook to `Colab > Auto Connect`
- optionally use `Upload to Colab` on this repository if you want to run your local uncommitted files
- then execute the setup cells below

In [ ]:
#@title 1. Prepare repository on the Colab runtime
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Yongjooon/QA-AI-FineTunning.git"
REPO_DIR = "/content/QA-AI-FineTunning"

if Path(REPO_DIR, "pyproject.toml").exists():
    print(f"Using existing repository on Colab runtime: {REPO_DIR}")
else:
    print(f"Repository not found on runtime. Cloning from {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Current working directory:", os.getcwd())

In [ ]:
#@title 2. Install dependencies
!pip -q install -U pip
!pip -q install -r requirements-colab.txt
!pip -q install -e .

In [ ]:
#@title 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 4. Inference parameters
MODEL_NAME = "google/gemma-3-4b-it"  # Must match the base model used in training
PERSIST_ROOT = "/content/drive/MyDrive/qa_ai_finetuning"
TRAIN_RUN_NAME = "gemma_qa_lora_run_01"
ADAPTER_PATH = f"{PERSIST_ROOT}/train_runs/{TRAIN_RUN_NAME}/final_model"
GENERATION_RUN_NAME = "gemma_qa_generation_01"
MAX_NEW_TOKENS = 1800
TEMPERATURE = 0.2
TOP_P = 0.9

In [ ]:
#@title 5. Inspect Colab runtime
import json
from qafinetune.runtime import detect_runtime

runtime_profile = detect_runtime()
print(json.dumps(runtime_profile, indent=2, ensure_ascii=False))

In [ ]:
#@title 6. Upload input zip for scenario generation
from google.colab import files
uploaded = files.upload()
INPUT_ZIP_PATH = next(iter(uploaded.keys()))
print("Uploaded:", INPUT_ZIP_PATH)

In [ ]:
#@title 7. Run generation
import subprocess

command = [
    "python", "-m", "qafinetune.infer",
    "--model_name", MODEL_NAME,
    "--adapter_path", ADAPTER_PATH,
    "--input_zip", INPUT_ZIP_PATH,
    "--output_root", PERSIST_ROOT,
    "--run_name", GENERATION_RUN_NAME,
    "--max_new_tokens", str(MAX_NEW_TOKENS),
    "--temperature", str(TEMPERATURE),
    "--top_p", str(TOP_P),
]
print(" ".join(command))
subprocess.run(command, check=True)

In [ ]:
#@title 8. Preview generated outputs
import json
from pathlib import Path

run_dir = Path(PERSIST_ROOT) / "generated_runs" / GENERATION_RUN_NAME
summary_path = run_dir / "scenario_summary.md"
json_path = run_dir / "playwright_scenario.json"

print("Summary path:", summary_path)
print(summary_path.read_text(encoding='utf-8'))
print("\nJSON path:", json_path)
print(json_path.read_text(encoding='utf-8')[:4000])